# Creating Your First Agent Society

Sổ tay này minh họa cách thiết lập và tận dụng khả năng của CAMEL để tạo ra xã hội agent đầu tiên của bạn thông qua lớp `RolePlaying()`.

**Agent Society:** Enabling multi-agent communication for the task solving.

## Philosophical Bits - Những suy ngẫm triết học

> Điều kỳ diệu nào khiến chúng ta trở nên thông minh? 
> 
> Điều kỳ diệu chính là không có điều kỳ diệu nào cả. 
> 
> Sức mạnh của trí tuệ bắt nguồn từ sự đa dạng rộng lớn của chúng ta, chứ không phải từ bất kỳ nguyên lý đơn lẻ và hoàn hảo nào.
> 
> — Marvin Minsky, The Society of Mind, trang 308


Trong phần này, chúng ta sẽ xem xét lớp RolePlaying() hướng đến nhiệm vụ. Chúng tôi thiết kế lớp này theo cách tuân thủ chỉ dẫn. Bản chất là để giải quyết một nhiệm vụ phức tạp, bạn có thể kích hoạt hai AI Agent giao tiếp cùng hợp tác làm việc từng bước để đạt được các giải pháp. Các khái niệm chính bao gồm:

- Task: một nhiệm vụ có thể đơn giản như một ý tưởng, được khởi tạo bởi một prompt khởi đầu.
- AI User: AI Agent được mong đợi sẽ cung cấp các chỉ dẫn.
- AI Assistant: tác nhân được kỳ vọng sẽ đưa ra các giải pháp đáp ứng các hướng dẫn.



In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
# Import necessary classes
from camel.societies import RolePlaying
from camel.types import TaskType, ModelType, ModelPlatformType
from camel.configs import ChatGPTConfig
from camel.models import ModelFactory

model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI,
    model_type=ModelType.GPT_4O_MINI,
    model_config_dict=ChatGPTConfig(temperature=0.0).as_dict(), # [Optional] the config for model
)

task_kwargs = {
    'task_prompt': 'Planning to build an AI application to help Vietnamese people overcome laziness and diligently learn new vocabulary every day.',
    'with_task_specify': True,
    'task_specify_agent_kwargs': {'model': model}
}

user_role_kwargs = {
    'user_role_name': 'A person with a burning desire to change Vietnamese education, make an English learning app the number one in Vietnam, and become a billionaire.',
    'user_agent_kwargs': {'model': model}
}

assistant_role_kwargs = {
    'assistant_role_name': 'An English teacher with deep understanding of psychology.',
    'assistant_agent_kwargs': {'model': model}
}

society = RolePlaying(
    **task_kwargs,             # The task arguments
    **user_role_kwargs,        # The instruction sender's arguments
    **assistant_role_kwargs,   # The instruction receiver's arguments
    output_language="Vietnamese"
)

/home/lai/Documents/camel-tutorial/.venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(
🖇 AgentOps: Could not record event - no sessions detected. Create a session by calling agentops.start_session()
🖇 AgentOps: Could not record event - no sessions detected. Create a session by calling agentops.start_session()
🖇 AgentOps: Could not record event - no sessions detected. Create a session by calling agentops.start_session()
🖇 AgentOps: Could not record event - no sessions detected. Create a session by calling agentops.start_session()
🖇 AgentOps: Could not record event - no sessions detected. Create a session by calling agentops.start_session()
🖇 AgentOps: Could not record event - no sessions detected. Create a session by calling agentops.start_session()
🖇 AgentOps: Could not record event - no sessions detected. Create a session by calling agen

In [3]:
def is_terminated(response):
    """
    Give alerts when the session should be terminated.
    """
    if response.terminated:
        role = response.msg.role_type.name
        reason = response.info['termination_reasons']
        print(f'AI {role} terminated due to {reason}')

    return response.terminated

In [4]:
def run(society, round_limit: int=10):

    # Get the initial message from the ai assistant to the ai user
    input_msg = society.init_chat()

    # Starting the interactive session
    for _ in range(round_limit):

        # Get the both responses for this round
        assistant_response, user_response = society.step(input_msg)

        # Check the termination condition
        if is_terminated(assistant_response) or is_terminated(user_response):
            break

        # Get the results
        print(f'[AI User] {user_response.msg.content}.\n')
        # Check if the task is end
        if 'CAMEL_TASK_DONE' in user_response.msg.content:
            break
        print(f'[AI Assistant] {assistant_response.msg.content}.\n')



        # Get the input message for the next round
        input_msg = assistant_response.msg

    return None

In [5]:
run(society)

[AI User] Instruction: Nghiên cứu thị trường để xác định nhu cầu và sở thích của người dùng Việt Nam đối với ứng dụng học từ vựng.  
Input: None.

[AI Assistant] Solution: Để nghiên cứu thị trường và xác định nhu cầu cũng như sở thích của người dùng Việt Nam đối với ứng dụng học từ vựng, chúng ta có thể thực hiện các bước sau:

1. **Khảo sát trực tuyến**: Tạo một bảng khảo sát trực tuyến với các câu hỏi liên quan đến thói quen học từ vựng, sở thích về hình thức học (trò chơi, video, bài tập), và các tính năng mong muốn (nhắc nhở, phần thưởng, cộng đồng). Sử dụng các nền tảng như Google Forms hoặc SurveyMonkey để thu thập dữ liệu.

2. **Phân tích đối thủ cạnh tranh**: Nghiên cứu các ứng dụng học từ vựng hiện có trên thị trường Việt Nam như Duolingo, Memrise, hay LingoDeer. Xem xét các tính năng, giao diện người dùng, và phản hồi từ người dùng để hiểu rõ hơn về những gì đang hoạt động tốt và những gì cần cải thiện.

3. **Phỏng vấn người dùng**: Tổ chức các buổi phỏng vấn với người dùng t